# Pythonizing Vincent!

In [1]:
import numpy as np
from typing import Union, Optional


# ------------------------------------------------------------
# 1) Ppi: build the orthogonal matrix V from first-order Pis
# ------------------------------------------------------------

def Ppi(Pi: Union[np.ndarray, list]) -> np.ndarray:
    """
    Python/Numpy port of the R function Ppi(Pi).

    Parameters
    ----------
    Pi : array-like of shape (N,)
        First-order inclusion probabilities, 0 < Pi_i < 1,
        sum(Pi) must be (approximately) an integer.

    Returns
    -------
    V : ndarray of shape (N, n)
        Orthogonal matrix associated to the DSD construction.
    """
    Pi = np.asarray(Pi, dtype=float).ravel()
    N = Pi.size

    # --- Error checks ---
    if N < 2:
        raise ValueError(
            "The sampling designs should be defined on a set of more than "
            "one element. (length(Pi) > 1)"
        )

    if np.any(Pi <= 0) or np.any(Pi >= 1):
        raise ValueError("Pi is not a vector of probabilities (0 < p < 1).")

    sum_pi_rounded = round(Pi.sum(), 9)
    n_int = int(sum_pi_rounded)
    if int(round(sum_pi_rounded, 9)) - sum_pi_rounded != 0:
        raise ValueError(
            "The sum of the first order inclusion probabilities "
            "should be an integer (up to rounding)."
        )

    # --- Main algorithm ---
    s_vals = np.zeros(N, dtype=float)
    c_vals = np.zeros(N, dtype=float)
    alpha = np.zeros(N, dtype=float)

    # kr will store the indices (0-based) where cumulative sum crosses integers
    if n_int <= 0:
        raise ValueError("Sum of Pi must be at least 1.")
    kr = [None] * n_int

    cum_sum = 0.0
    r = 1          # current integer threshold
    r_prev = 0     # last integer that was crossed

    for k in range(N):
        prev_sum = cum_sum
        cum_sum += Pi[k]

        if cum_sum >= r:  # crossed integer r
            if r <= n_int:
                alpha[k] = r - prev_sum
                kr[r - 1] = k   # store 0-based index
                val = np.sqrt((1.0 - Pi[k]) / (1.0 - alpha[k]))
                s_vals[k] = np.round(val, 8)
                r_prev = r
                r += 1
        else:
            denom = (r_prev + 1 - prev_sum)
            val = np.sqrt(Pi[k] / denom)
            s_vals[k] = np.round(val, 15)

        c_vals[k] = np.sqrt(1.0 - s_vals[k] ** 2)

    # Patch: ensure last crossing index corresponds to last unit (like R hack)
    # If some entries of kr are still None, set the last one to N-1.
    if any(x is None for x in kr):
        kr[-1] = N - 1
    # For safety, also replace any remaining None by the last index
    last_index = kr[-1]
    kr = [last_index if x is None else x for x in kr]

    # Use sample size n_int as number of columns (simplified vs. R hack)
    r_prev = n_int

    # --- Build V ---
    V = np.zeros((N, r_prev), dtype=float)
    V[0, 0] = 1.0

    # In R: V[kr[r] + 1, r + 1] = 1 for r in 1:(r_prev-1)
    # Here r_idx corresponds to r-1 in R, and we use 0-based indices.
    if r_prev - 1 != 0:
        for r_idx in range(1, r_prev):
            kpos = kr[r_idx - 1]  # 0-based index for row
            V[kpos + 1, r_idx] = 1.0

    # Apply Givens-like rotations
    for k in range(N - 1):
        L = V[k, :].copy()
        M = V[k + 1, :].copy()
        V[k, :] = s_vals[k] * L - c_vals[k] * M
        V[k + 1, :] = c_vals[k] * L + s_vals[k] * M

    return V


# ------------------------------------------------------------
# 2) DSD sampling (real and complex versions)
# ------------------------------------------------------------

def Drawing_Dsd(
    v: Union[np.ndarray, list],
    s: int = 1,
    B: bool = False,
    seed: Optional[int] = None,
):
    """
    Python/Numpy port of the R function Drawing_Dsd(v, s=1, B=FALSE, seed=NULL).

    Parameters
    ----------
    v : array-like, shape (N, n)
        Matrix of (real or complex) vectors used in DSD.
    s : int, default 1
        Number of samples (replicates).
    B : bool, default False
        If True: return 0/1 indicator vector(s).
        If False: return indices of selected units (1-based, to match R).
    seed : int or None
        Seed for reproducibility.

    Returns
    -------
    If s == 1:
        1D array of length N (if B=True) or selected indices (if B=False).
    If s > 1:
        2D array of shape (N, s) (if B=True) or (n, s) with indices per sample.
    """
    v = np.asarray(v)
    rng = np.random.default_rng(seed)

    if np.iscomplexobj(v):
        return _dsd_sampling_mult_complex(v, s, B, rng)
    else:
        return _dsd_sampling_mult(v, s, B, rng)


# ---------------------- helpers: real case ---------------------- #

def _dsd_sampling_mult(
    v: np.ndarray,
    s: int,
    B: bool,
    rng: np.random.Generator,
):
    v = np.asarray(v, dtype=float)
    if v.ndim == 1:
        v = v[:, None]  # treat as (N,1)

    if s == 1:
        return _dsd_sampling_01_B_C(v, B, rng)
    else:
        samples = [_dsd_sampling_01_B_C(v, B, rng) for _ in range(s)]
        # stack as columns like R's replicate (N x s)
        return np.column_stack(samples)


def _dsd_sampling_01_B_C(
    v: np.ndarray,
    B: bool = True,
    rng: Optional[np.random.Generator] = None,
):
    """
    Real-valued version of .dsd_sampling_01_B_C in R.
    """
    v = np.asarray(v, dtype=float)
    if v.ndim == 1:
        v = v[:, None]

    N, n = v.shape
    echant = np.zeros(N, dtype=int)

    if rng is None:
        rng = np.random.default_rng()
    ref = rng.random(n)

    # Step 1: first element
    w = v.copy()

    pi1 = np.einsum("ij,ij->i", v, v)  # diag(v %*% t(v))
    total = 0.0
    i = -1

    while total < ref[0]:
        i += 1
        if i >= N:
            raise RuntimeError("Sampling failed in first step (real case).")
        total += pi1[i] / n
    echant[i] = 1

    l = v[i, :]
    norm_l = np.sqrt(np.dot(l, l))
    if norm_l == 0:
        raise ValueError("Encountered zero-norm vector in real DSD.")
    e1 = l / norm_l

    # Step 2: remaining n-1 elements
    for j in range(n - 1):
        r = n - (j + 1)
        inter = v @ e1  # (N,)
        pi1 = pi1 - inter * inter
        pi2 = pi1 / r

        total = 0.0
        i = -1
        while total < ref[j + 1]:
            i += 1
            if i >= N:
                raise RuntimeError("Sampling failed in step 2 (real case).")
            total += pi2[i]
        echant[i] = 1

        # Update w and e1 (Gram-Schmidt-like update)
        proj = w @ e1                      # shape (N,)
        w = w - np.outer(proj, e1)         # (N,n)
        L = w[i, :]
        norm_L = np.sqrt(np.dot(L, L))
        if norm_L == 0:
            raise ValueError("Encountered zero-norm vector in real DSD.")
        e1 = L / norm_L

    if B:
        # 0/1 vector -> same as R "echant"
        return echant
    else:
        # indices (1-based, like in R: (1:N)[echant==1])
        return np.nonzero(echant == 1)[0] + 1


# -------------------- helpers: complex case --------------------- #

def _dsd_sampling_mult_complex(
    v: np.ndarray,
    s: int,
    B: bool,
    rng: np.random.Generator,
):
    v = np.asarray(v, dtype=np.complex128)
    if v.ndim == 1:
        v = v[:, None]

    if s == 1:
        return _dsd_sampling_01_B_C_complex(v, B, rng)
    else:
        samples = [_dsd_sampling_01_B_C_complex(v, B, rng) for _ in range(s)]
        return np.column_stack(samples)


def _dsd_sampling_01_B_C_complex(
    v: np.ndarray,
    B: bool = True,
    rng: Optional[np.random.Generator] = None,
):
    """
    Complex-valued version of .dsd_sampling_01_B_C_complex in R.
    """
    v = np.asarray(v, dtype=np.complex128)
    if v.ndim == 1:
        v = v[:, None]

    N, n = v.shape
    echant = np.zeros(N, dtype=int)

    if rng is None:
        rng = np.random.default_rng()
    ref = rng.random(n)

    w = v.copy()

    # pi1 = Re( diag( v %*% t(Conj(v)) ) )
    pi1 = np.real(np.einsum("ij,ij->i", v, np.conjugate(v)))

    if np.any(pi1 < 0) or np.any(pi1 >= 1):
        raise ValueError(
            "The matrix v given as input doesn't suit the expected input "
            "(cf. pgd / periodic_dsd)."
        )

    # Step 1: first element
    total = 0.0
    i = -1

    while total < ref[0]:
        i += 1
        if i >= N:
            raise RuntimeError("Sampling failed in first step (complex case).")
        total += pi1[i] / n
    echant[i] = 1

    M = v[i, :]
    norm_M = np.sqrt(np.real(np.vdot(M, M)))
    if norm_M == 0:
        raise ValueError("Encountered zero-norm vector in complex DSD.")
    e1 = M / norm_M

    # Step 2: remaining n-1 elements
    for j in range(n - 1):
        r = n - (j + 1)

        # inter <- v %*% Conj(e1)
        inter = v @ np.conjugate(e1)           # (N,)
        pi1 = pi1 - np.real(inter * np.conjugate(inter))
        pi2 = np.real(pi1 / r)

        total = 0.0
        i = -1
        while total < ref[j + 1]:
            i += 1
            if i >= N:
                raise RuntimeError("Sampling failed in step 2 (complex case).")
            total += pi2[i]
        echant[i] = 1

        # w <- w - (projection on e1)
        proj = w @ np.conjugate(e1)           # (N,)
        w = w - np.outer(proj, e1)            # (N, n)

        L = w[i, :]
        norm_L = np.sqrt(np.real(np.vdot(L, L)))
        if norm_L == 0:
            raise ValueError("Encountered zero-norm vector in complex DSD.")
        e1 = L / norm_L

    if B:
        return echant
    else:
        # Return 1-based indices, to match the R function
        return np.nonzero(echant == 1)[0] + 1

In [2]:
import numpy as np
from typing import Optional, Union


def spec(omega: np.ndarray, M: int, pi: Union[np.ndarray, list], spectre=100) -> np.ndarray:
    """
    Python/Numpy port of the R function:

        spec <- function(omega, M, pi, spectre = 100)

    Parameters
    ----------
    omega : array-like, shape (M, N)
        Omega matrix (same role as in the R code).
    M : int
        Integer (usually sum(pi)).
    pi : array-like, shape (N,)
        First-order inclusion probabilities.
    spectre : 100 or array-like of length M
        If 100 (default), the spectrum is constructed as in the R code.
        Otherwise, used directly as the initial spectrum.

    Returns
    -------
    mat_spectre : ndarray, shape (M, N)
    """
    pi = np.asarray(pi, dtype=float).ravel()
    N = pi.size
    omega = np.asarray(omega, dtype=float)
    if omega.shape != (M, N):
        raise ValueError("omega must have shape (M, N)")

    mat_spectre = np.zeros((M, N), dtype=float)
    pi_down = np.sort(pi)[::-1]
    mu = pi.sum()

    # --- Build initial spectrum (last column) if needed ---
    if (np.isscalar(spectre) and spectre == 100) or (
        not np.isscalar(spectre)
        and len(np.atleast_1d(spectre)) == 1
        and np.atleast_1d(spectre)[0] == 100
    ):
        spectre_vec = np.zeros(M, dtype=float)
        cumsum_1 = 0.0
        lam = 0.0

        # j in 1:(M-1)  ->  jR = j+1
        for j in range(M - 1):
            jR = j + 1
            A = max(lam, mu - cumsum_1 - (M - jR))
            B = 1.0

            # i in 1:(M-jR)
            for iR in range(1, M - jR + 1):
                # sum(pi_down[1:(M-j-i+1)]) in R
                t_idx = M - jR - iR + 1  # always >= 1 here
                s_down = pi_down[:t_idx].sum()
                B = min(B, (mu - cumsum_1 - s_down) / iR)

            B = min(B, (mu - cumsum_1) / (M - jR + 1))
            spectre_vec[j] = A + omega[j, N - 1] * (B - A)
            lam = spectre_vec[j]
            cumsum_1 += spectre_vec[j]

        # last component
        spectre_vec[M - 1] = mu - spectre_vec[: M - 1].sum()

    else:
        spectre_arr = np.asarray(spectre, dtype=float).ravel()
        if spectre_arr.size != M:
            raise ValueError("spectre must have length M")
        spectre_vec = spectre_arr

    mat_spectre[:, N - 1] = spectre_vec

    # --- Backward recursion for other columns (k in (N-1):1) ---
    for kR in range(N - 1, 0, -1):  # kR = N-1, ..., 1
        startR = max(1, M - kR + 1)
        lambda1 = mat_spectre[:, kR].copy()     # column k+1 in R
        lambda2 = mat_spectre[:, kR - 1].copy() # column k in R

        # j in start:M
        for jR in range(startR, M + 1):
            # A = max(0, lambda1[j-1], sum(lambda1[1:j]) - sum(lambda2[0:(j-1)]) - pi_down[k+1])
            if jR >= 2:
                lam_prev = lambda1[jR - 2]
            else:
                lam_prev = 0.0

            sum_l1 = lambda1[:jR].sum()
            sum_l2 = lambda2[: jR - 1].sum() if jR > 1 else 0.0
            A = max(0.0, lam_prev, sum_l1 - sum_l2 - pi_down[kR])  # pi_down[k+1] -> index kR

            # Build B_1
            B_list = []
            for iR in range(jR, M + 1):
                prem = 0.0
                deux = 0.0
                trois = lambda2[: jR - 1]

                # prem = pi_down[(M-i+1):k] if (M-i+1) <= k
                if (M - iR + 1) <= kR:
                    start_idx = M - iR + 1  # 1-based
                    prem = pi_down[start_idx - 1 : kR].sum()

                # deux = lambda1[j:(i-1)] if j <= i-1
                if jR <= (iR - 1):
                    deux = lambda1[jR - 1 : iR - 1].sum()

                B_val = prem - deux - trois.sum()
                B_list.append(B_val)

            B_inner = min(B_list) if B_list else float("inf")
            B = min(lambda1[jR - 1], B_inner)

            mat_spectre[jR - 1, kR - 1] = A + omega[jR - 1, kR - 1] * (B - A)
            lambda2[jR - 1] = mat_spectre[jR - 1, kR - 1]

    return mat_spectre


def CaDsd(
    pi: Union[np.ndarray, list],
    M: Optional[int] = None,
    omega: Optional[np.ndarray] = None,
    rho: Optional[np.ndarray] = None,
    spectre=100,
    U: Optional[np.ndarray] = None,
    option: bool = True,
):
    """
    Python/Numpy port of the R function:

        CaDsd <- function(omega=matrix(0.5,M,length(pi)),
                          rho=matrix(0.5,M,length(pi)-1),
                          M=round(sum(pi),7),
                          pi,
                          spectre=100,
                          U=diag(M),
                          option=TRUE)

    Parameters
    ----------
    pi : array-like, length N
    M : int or None
        If None, set to round(sum(pi), 7) as in R.
    omega : array-like, shape (M, N), default 0.5
    rho : array-like, shape (M, N-1), default 0.5
    spectre : passed to spec(...)
    U : initial unitary matrix, shape (M, M); default identity.
    option : kept for signature compatibility (not used here).

    Returns
    -------
    dict with keys: "K", "spectrum", "EigenBasis"
    """
    pi = np.asarray(pi, dtype=float).ravel()
    N = pi.size

    # M default
    if M is None:
        M = int(round(pi.sum(), 7))
    if int(M) != M:
        raise ValueError("M should be an integer")
    M = int(M)

    # Defaults for omega and rho
    if omega is None:
        omega = 0.5 * np.ones((M, N), dtype=float)
    else:
        omega = np.asarray(omega, dtype=float)
        if omega.shape != (M, N):
            raise ValueError("omega must have shape (M, N)")

    if rho is None:
        rho = 0.5 * np.ones((M, N - 1), dtype=float)
    else:
        rho = np.asarray(rho, dtype=float)
        if rho.shape != (M, N - 1):
            raise ValueError("rho must have shape (M, N-1)")

    # Convert rho to angles and compute spectrum
    rho = np.round(rho * 2 * np.pi, 7)
    mat_spectre = np.round(spec(omega, M, pi, spectre), 7)

    pi_down = np.sort(pi)[::-1]

    # U default
    if U is None:
        U = np.eye(M, dtype=complex)
    else:
        U = np.asarray(U, dtype=complex)
        if U.shape != (M, M):
            raise ValueError("U must have shape (M, M)")

    # phi = sqrt(pi_down[1])*U[,1] (1st column, 1-based)
    phi = np.round(np.sqrt(pi_down[0]) * U[:, [0]], 7)  # shape (M, 1)
    ens = np.arange(1, M + 1, dtype=int)               # 1..M

    # Main loop over k = 2..N
    for kR in range(2, N + 1):
        # V = diag(complex(argument = rho[,k-1], modulus = 1))
        rho_col = rho[:, kR - 2]
        V = np.diag(np.exp(1j * rho_col))

        lambda1 = mat_spectre[:, kR - 1].copy()
        lambda2 = mat_spectre[:, kR - 2].copy()

        E1 = ens.copy().tolist()
        E2 = ens.copy().tolist()

        # Matching equal eigenvalues between lambda2 and lambda1
        for jR in ens:
            val = lambda2[jR - 1]
            if not E1:
                break
            lam1_E1 = lambda1[np.array(E1) - 1]
            matches = np.where(lam1_E1 == val)[0]
            if matches.size > 0:
                # remove jR from E2
                E2 = [x for x in E2 if x != jR]
                # remove the first match from E1
                del E1[matches[0]]

        E1_arr = np.array(E1, dtype=int)
        E2_arr = np.array(E2, dtype=int)
        E1_ = M + 1 - E1_arr
        E2_ = M + 1 - E2_arr
        r = len(E1_)

        # Permutation matrices sigma1, sigma2
        if r != M:
            E1_c = ens[~np.isin(ens, E1_)]
            E2_c = ens[~np.isin(ens, E2_)]

            E1_ = np.concatenate([np.sort(E1_), np.sort(E1_c)])
            E2_ = np.concatenate([np.sort(E2_), np.sort(E2_c)])

            sigma1 = np.eye(M, dtype=complex)[E1_ - 1, :]
            sigma2 = np.eye(M, dtype=complex)[E2_ - 1, :]
        else:
            sigma1 = np.eye(M, dtype=complex)
            sigma2 = np.eye(M, dtype=complex)

        if r != 0:
            # R = diag(r)[r:1,] %*% cbind(lambda2[E2], lambda1[E1])
            lambda2_E2 = lambda2[E2_arr - 1]
            lambda1_E1 = lambda1[E1_arr - 1]
            R = np.column_stack([lambda2_E2, lambda1_E1]).astype(complex)
            R = R[::-1, :]  # reverse rows

            v = np.zeros(r, dtype=complex)
            w = np.zeros(r, dtype=complex)

            for i in range(r):
                # v-part
                v1 = R[i, 0] - R[:, 1]
                v2 = R[i, 0] - R[:, 0]
                v2[i] = 1.0 + 0j
                order = np.argsort(np.abs(v1))
                v1s = v1[order]
                v2s = v2[order]
                ratio_prod = np.prod(v1s / v2s)
                v[i] = np.round(np.sqrt(-ratio_prod), 7)

                # w-part
                w1 = R[i, 1] - R[:, 0]
                w2 = R[i, 1] - R[:, 1]
                w2[i] = 1.0 + 0j
                order_w = np.argsort(np.abs(w1))
                w1s = w1[order_w]
                w2s = w2[order_w]
                ratio_prod_w = np.prod(w1s / w2s)
                w[i] = np.round(np.sqrt(ratio_prod_w), 7)

            # vect[1:r] = v  (1-based)
            vect = np.zeros((M, 1), dtype=complex)
            vect[:r, 0] = v

            # W = (1/(t(matrix(R[,2],r,r)) - matrix(R[,1],r,r))) * (v %*% t(w))
            col_vals = R[:, 1]
            row_vals = R[:, 0]
            denom = col_vals[np.newaxis, :] - row_vals[:, np.newaxis]  # shape (r,r)
            W = (v[:, None] * w[None, :]) / denom

            # phi <- cbind(phi, U %*% V %*% t(sigma2) %*% vect)
            temp = U @ V @ sigma2.T @ vect  # (M,1)
            phi = np.hstack([phi, temp])

            # U <- U %*% V %*% t(sigma2) %*% rbind(mat1,mat2) %*% sigma1
            mat1 = np.hstack([W, np.zeros((r, M - r), dtype=complex)])
            mat2 = np.hstack([np.zeros((M - r, r), dtype=complex),
                              np.eye(M - r, dtype=complex)])
            block = np.vstack([mat1, mat2])
            U = U @ V @ sigma2.T @ block @ sigma1

    # EigenBasis <- t(Conj(phi)) %*% U %*% solve(sqrt(diag(mat_spectre[,N])))
    d = np.sqrt(mat_spectre[:, N - 1])
    if np.any(d == 0):
        raise ValueError("Zero on diagonal of spectrum, cannot invert sqrt.")
    inv_sqrt_diag = np.diag(1.0 / d).astype(complex)

    EigenBasis = phi.conj().T @ U @ inv_sqrt_diag

    # K = t(Conj(phi)) %*% phi
    K = np.round(phi.conj().T @ phi, 7)

    return {
        "K": K,                   # Gram matrix
        "spectrum": mat_spectre,  # spectrum matrix
        "EigenBasis": EigenBasis  # eigenbasis
    }

In [3]:
import numpy as np

# --------------------------------------------------------
# 1) inclusionprobabilities() – Python version
#    (size measure p -> inclusion probs π with sum π = n)
# --------------------------------------------------------

def inclusionprobabilities(p, n):
    """
    Rough equivalent of sampling::inclusionprobabilities(p, n) in R.

    p : array-like, strictly positive size measures
    n : desired fixed sample size (integer)

    Returns
    -------
    pik : ndarray of length N
        First-order inclusion probabilities, 0 < pik_i <= 1, sum pik_i = n.
    """
    p = np.asarray(p, dtype=float)
    N = p.size
    if n <= 0 or n > N:
        raise ValueError("n must be in 1..N")

    pik = np.zeros(N, dtype=float)
    mask_fixed = np.zeros(N, dtype=bool)  # True where pik is already fixed to 1
    n_rem = float(n)
    p_work = p.copy()

    # Iterative rescaling: set any prob >= 1 to 1, rescale remaining
    while True:
        idx = ~mask_fixed
        if not np.any(idx):
            break

        p_sub = p_work[idx]
        # if all remaining p_sub are zero but n_rem > 0, it's impossible
        if p_sub.sum() <= 0 and n_rem > 1e-12:
            raise RuntimeError("Cannot construct inclusion probabilities with given p and n.")

        temp = n_rem * p_sub / p_sub.sum()
        over = temp >= (1.0 - 1e-12)  # tolerance

        # If no temp >= 1, we're done
        if not np.any(over):
            pik[idx] = temp
            break

        # Fix those >=1 to exactly 1
        idx_global = np.where(idx)[0]
        over_global = idx_global[over]

        pik[over_global] = 1.0
        mask_fixed[over_global] = True
        n_rem -= over.sum()
        p_work[over_global] = 0.0

        if n_rem <= 1e-12:
            # All remaining inclusion probabilities must be zero
            break

    return pik




In [4]:
def to_r_vector(arr, name="vec"):
    arr = np.asarray(arr).ravel()
    values = ", ".join([f"{x:.10f}" for x in arr])
    return f"{name} <- c({values})"

def to_r_matrix_complex(mat, name="mat"):
    mat = np.asarray(mat)
    N, M = mat.shape
    
    r_code = f"{name} <- matrix(c(\n"
    elements = []
    
    for j in range(M):  # R fills by column
        for i in range(N):
            val = mat[i, j]
            real_part = np.real(val)
            imag_part = np.imag(val)
            
            if abs(imag_part) < 1e-10:
                elements.append(f"  {real_part:.10f}")
            else:
                sign = "+" if imag_part >= 0 else ""
                elements.append(f"  complex(real={real_part:.10f}, imaginary={imag_part:.10f})")
    
    r_code += ",\n".join(elements)
    r_code += f"\n), nrow={N}, ncol={M}, byrow=FALSE)"
    return r_code

def to_r_matrix_real(mat, name="mat"):
    mat = np.asarray(mat)
    N, M = mat.shape
    
    r_code = f"{name} <- matrix(c(\n"
    elements = []
    
    for j in range(M):  # R fills by column
        for i in range(N):
            elements.append(f"  {mat[i, j]:.10f}")
    
    r_code += ",\n".join(elements)
    r_code += f"\n), nrow={N}, ncol={M}, byrow=FALSE)"
    return r_code


## Update notes

This version was reorganized so that the sensitivity analysis is at the end of the notebook.

Main changes:
- ABC progress printing is now concise and persistent: it reports at a controlled interval, not every iteration.
- The ABC onlooker phase was corrected to use a full onlooker population instead of usually updating only about one source per iteration.
- Added elitism, adaptive step size, optional local search around the best source, and stronger scout replacement.
- Added internal ordering alignment for `CaDsd`: because `CaDsd` returns the kernel diagonal in descending `π` order, the vectors used in the variance calculation are aligned to that order inside `ABCAlgorithm`.
- ABC results are stored in `abc_run_details`, so the final sensitivity section can analyze the last or selected run.


# ABC Algorithm (improved)


In [5]:
import numpy as np
import time
from typing import Tuple, List, Optional, Dict, Any


class ABCAlgorithm:
    """
    Improved Artificial Bee Colony optimizer for the CaDsd parameterization.

    Important implementation note
    -----------------------------
    CaDsd internally sorts inclusion probabilities in descending order and returns
    K with diag(K) in that descending-pi order. To keep the Horvitz-Thompson
    variance calculation consistent, this class aligns y, z, and pi to that same
    order by default. Set enforce_cadsd_order=False only if you intentionally
    manage the ordering outside the class.
    """

    def __init__(
        self,
        y_sorted,
        z_sorted,
        pik_sorted,
        var_srs_y,
        var_srs_z,
        M,
        n,
        case_name="",
        objective: str = "eff_z",
        enforce_cadsd_order: bool = True,
        random_state: Optional[int] = None,
    ):
        self.rng = np.random.default_rng(random_state)

        self.input_y = np.asarray(y_sorted, dtype=float).ravel()
        self.input_z = np.asarray(z_sorted, dtype=float).ravel()
        self.input_pi = np.asarray(pik_sorted, dtype=float).ravel()

        if not (len(self.input_y) == len(self.input_z) == len(self.input_pi)):
            raise ValueError("y_sorted, z_sorted, and pik_sorted must have the same length.")

        self.enforce_cadsd_order = enforce_cadsd_order
        if enforce_cadsd_order:
            self.order = np.argsort(self.input_pi)[::-1]
        else:
            self.order = np.arange(len(self.input_pi))

        self.y_sorted = self.input_y[self.order]
        self.z_sorted = self.input_z[self.order]
        self.pik_sorted = self.input_pi[self.order]

        self.var_srs_y = float(var_srs_y)
        self.var_srs_z = float(var_srs_z)
        self.M = int(M)
        self.n = int(n)
        self.N = len(self.pik_sorted)
        self.case_name = case_name
        self.objective = objective

        self.I_N = np.eye(self.N)
        self.Dpi_inv = np.diag(1.0 / self.pik_sorted)

        # CaDsd validation target. With enforce_cadsd_order=True this is equal
        # to self.pik_sorted, but this explicit vector is safer.
        self.pik_sorted_desc = np.sort(self.input_pi)[::-1]

        self._calculate_optimal()

        # Best solution tracking
        self.global_best_eff_z = 0.0
        self.global_best_eff_y = 0.0
        self.global_best_score = -np.inf
        self.global_best_omega = None
        self.global_best_rho = None

        # History and counters
        self.history = []              # numeric best eff_z, kept for backward compatibility
        self.history_records = []      # richer records for diagnostics
        self.scout_history = []
        self.eval_count = 0
        self.valid_count = 0
        self.best_update_count = 0

    def _calculate_optimal(self):
        """Compute the reference P_pi design efficiency."""
        Base_opt = Ppi(self.pik_sorted)
        Ppi_mat = Base_opt @ Base_opt.T
        A_opt = (self.I_N - Ppi_mat) * Ppi_mat

        var_opt_y = float(np.real(self.y_sorted.T @ self.Dpi_inv @ A_opt @ self.Dpi_inv @ self.y_sorted))
        var_opt_z = float(np.real(self.z_sorted.T @ self.Dpi_inv @ A_opt @ self.Dpi_inv @ self.z_sorted))

        self.var_opt_y = var_opt_y
        self.var_opt_z = var_opt_z
        self.eff_y_optimal = self.var_srs_y / var_opt_y if var_opt_y > 0 else np.inf
        self.eff_z_optimal = self.var_srs_z / var_opt_z if var_opt_z > 0 else np.inf

    def _score(self, eff_z: float, eff_y: float) -> float:
        """Objective used by ABC. Default is z-efficiency."""
        if self.objective == "eff_z":
            return eff_z
        if self.objective == "eff_y":
            return eff_y
        if self.objective == "harmonic":
            if eff_z <= 0 or eff_y <= 0:
                return 0.0
            return 2.0 * eff_z * eff_y / (eff_z + eff_y)
        if self.objective == "mean":
            return 0.5 * (eff_z + eff_y)
        raise ValueError("objective must be one of: 'eff_z', 'eff_y', 'harmonic', 'mean'.")

    def evaluate(self, omega: np.ndarray, rho: np.ndarray) -> Tuple[float, float, bool]:
        """
        Evaluate a solution.

        Returns
        -------
        eff_z, eff_y, valid
        """
        self.eval_count += 1

        try:
            K_dict = CaDsd(pi=self.pik_sorted, M=self.M, omega=omega, rho=rho)
            Kmat = K_dict["K"].astype(np.complex128)
        except Exception:
            return (0.0, 0.0, False)

        diag_K = np.real(np.diag(Kmat))

        if not np.allclose(diag_K, self.pik_sorted_desc, atol=1e-3):
            return (0.0, 0.0, False)

        try:
            evals = np.linalg.eigvalsh(Kmat)
        except Exception:
            return (0.0, 0.0, False)

        if not (np.all(evals >= -1e-4) and np.all(evals <= 1 + 1e-4)):
            return (0.0, 0.0, False)
        if not np.isclose(evals.sum(), self.n, atol=1e-4):
            return (0.0, 0.0, False)

        A = Kmat * (self.I_N - Kmat.conj())
        var_y = float(np.real(self.y_sorted.conj().T @ self.Dpi_inv @ A @ self.Dpi_inv @ self.y_sorted))
        var_z = float(np.real(self.z_sorted.conj().T @ self.Dpi_inv @ A @ self.Dpi_inv @ self.z_sorted))

        if (not np.isfinite(var_y)) or (not np.isfinite(var_z)) or var_y <= 0 or var_z <= 0:
            return (0.0, 0.0, False)

        eff_y = self.var_srs_y / var_y
        eff_z = self.var_srs_z / var_z

        if not (np.isfinite(eff_y) and np.isfinite(eff_z)):
            return (0.0, 0.0, False)

        self.valid_count += 1
        return (eff_z, eff_y, True)

    def _food_from_arrays(self, omega: np.ndarray, rho: np.ndarray, trial: int = 0) -> Optional[dict]:
        eff_z, eff_y, valid = self.evaluate(omega, rho)
        if not valid:
            return None
        score = self._score(eff_z, eff_y)
        food = {
            "omega": omega,
            "rho": rho,
            "eff_z": eff_z,
            "eff_y": eff_y,
            "score": score,
            "trial": trial,
        }
        self._update_global_best(food)
        return food

    def _update_global_best(self, food: dict) -> bool:
        if food["score"] > self.global_best_score:
            self.global_best_score = food["score"]
            self.global_best_eff_z = food["eff_z"]
            self.global_best_eff_y = food["eff_y"]
            self.global_best_omega = food["omega"].copy()
            self.global_best_rho = food["rho"].copy()
            self.best_update_count += 1
            return True
        return False

    def _random_candidate(self) -> Tuple[np.ndarray, np.ndarray]:
        omega = self.rng.random((self.M, self.N))
        rho = self.rng.random((self.M, self.N - 1))
        return omega, rho

    def _candidate_near_best(self, scale: float = 0.05) -> Tuple[np.ndarray, np.ndarray]:
        if self.global_best_omega is None or self.global_best_rho is None:
            return self._random_candidate()
        omega = self.global_best_omega + self.rng.normal(0.0, scale, self.global_best_omega.shape)
        rho = self.global_best_rho + self.rng.normal(0.0, scale, self.global_best_rho.shape)
        return np.clip(omega, 0.0, 1.0), np.clip(rho, 0.0, 1.0)

    def initialize_population(self, colony_size: int, verbose: bool = True) -> List[dict]:
        """Initialize food sources."""
        if verbose:
            print(f" Initializing {colony_size} food sources...")

        population = []

        # Add the center point first. It is often a useful valid baseline.
        center_omega = 0.5 * np.ones((self.M, self.N))
        center_rho = 0.5 * np.ones((self.M, self.N - 1))
        food = self._food_from_arrays(center_omega, center_rho)
        if food is not None:
            population.append(food)

        max_attempts = max(colony_size * 30, 200)
        count = 0

        while len(population) < colony_size and count < max_attempts:
            omega, rho = self._random_candidate()
            food = self._food_from_arrays(omega, rho)
            if food is not None:
                population.append(food)

            count += 1
            if verbose and count % 100 == 0:
                print(f"   evaluated {count}, valid sources {len(population)}/{colony_size}", end="\r")

        if verbose:
            print(f"\n    Initialized {len(population)} food sources (requested {colony_size})")
            if len(population) > 0:
                print(f"    Best initial: eff_z={self.global_best_eff_z:.4f}, eff_y={self.global_best_eff_y:.4f}")
            else:
                print("    WARNING: no valid solution found.")

        return population

    def _mutate_food(
        self,
        food: dict,
        partner: dict,
        progress: float,
        mode: str = "abc",
    ) -> Tuple[np.ndarray, np.ndarray]:
        """
        Create a new candidate.

        Improvements over the first version:
        - adaptive scale decreases during the run;
        - optional pull toward the global best;
        - small Gaussian jitter prevents early stagnation.
        """
        # High at the start, lower near the end.
        adaptive_scale = max(0.15, 1.0 - 0.85 * progress)

        if mode == "local":
            return self._candidate_near_best(scale=0.02 + 0.08 * (1.0 - progress))

        phi_omega = self.rng.uniform(-adaptive_scale, adaptive_scale, food["omega"].shape)
        phi_rho = self.rng.uniform(-adaptive_scale, adaptive_scale, food["rho"].shape)

        new_omega = food["omega"] + phi_omega * (food["omega"] - partner["omega"])
        new_rho = food["rho"] + phi_rho * (food["rho"] - partner["rho"])

        # Mild attraction to global best after the early exploration period.
        if self.global_best_omega is not None and progress > 0.10:
            pull = self.rng.uniform(0.0, 0.35 * progress)
            new_omega = new_omega + pull * (self.global_best_omega - new_omega)
            new_rho = new_rho + pull * (self.global_best_rho - new_rho)

        # Tiny jitter helps escape identical/stagnant sources.
        jitter = 0.01 * adaptive_scale
        new_omega = new_omega + self.rng.normal(0.0, jitter, new_omega.shape)
        new_rho = new_rho + self.rng.normal(0.0, jitter, new_rho.shape)

        return np.clip(new_omega, 0.0, 1.0), np.clip(new_rho, 0.0, 1.0)

    def _greedy_replace(self, old_food: dict, omega: np.ndarray, rho: np.ndarray) -> dict:
        new_food = self._food_from_arrays(omega, rho)
        if new_food is not None and new_food["score"] > old_food["score"]:
            return new_food
        old_copy = old_food.copy()
        old_copy["trial"] = old_food["trial"] + 1
        return old_copy

    def employed_bee_phase(self, population: List[dict], progress: float) -> List[dict]:
        if len(population) == 0:
            return population

        new_population = []
        for i, food in enumerate(population):
            if len(population) > 1:
                choices = np.delete(np.arange(len(population)), i)
                k = int(self.rng.choice(choices))
            else:
                k = i

            new_omega, new_rho = self._mutate_food(food, population[k], progress, mode="abc")
            new_population.append(self._greedy_replace(food, new_omega, new_rho))

        return new_population

    def onlooker_bee_phase(self, population: List[dict], progress: float) -> List[dict]:
        """
        Corrected onlooker phase.

        The previous version tested each food source with probability p_i, so because
        sum(p_i)=1 it typically generated only about one onlooker move per iteration.
        Standard ABC uses approximately one onlooker bee per food source.
        """
        if len(population) == 0:
            return population

        scores = np.array([food["score"] for food in population], dtype=float)
        scores = scores - np.min(scores) + 1e-12
        if not np.isfinite(scores).all() or scores.sum() <= 0:
            probs = np.ones(len(population)) / len(population)
        else:
            probs = scores / scores.sum()

        new_population = [food.copy() for food in population]

        for _ in range(len(population)):
            i = int(self.rng.choice(len(population), p=probs))
            if len(population) > 1:
                choices = np.delete(np.arange(len(population)), i)
                k = int(self.rng.choice(choices))
            else:
                k = i

            mode = "local" if self.rng.random() < 0.15 else "abc"
            new_omega, new_rho = self._mutate_food(new_population[i], population[k], progress, mode=mode)
            new_population[i] = self._greedy_replace(new_population[i], new_omega, new_rho)

        return new_population

    def scout_bee_phase(self, population: List[dict], limit: int, progress: float) -> Tuple[List[dict], int]:
        if len(population) == 0:
            return population, 0

        new_population = []
        n_abandoned = 0

        for food in population:
            if food["trial"] >= limit:
                n_abandoned += 1

                best_new = None
                attempts = 0
                max_attempts = 80

                while attempts < max_attempts:
                    # Half local around the current best, half fresh random.
                    if self.rng.random() < 0.50:
                        omega, rho = self._candidate_near_best(scale=0.04 + 0.10 * (1.0 - progress))
                    else:
                        omega, rho = self._random_candidate()

                    candidate = self._food_from_arrays(omega, rho)
                    if candidate is not None:
                        if best_new is None or candidate["score"] > best_new["score"]:
                            best_new = candidate

                    attempts += 1

                if best_new is not None:
                    best_new["trial"] = 0
                    new_population.append(best_new)
                else:
                    # Keep old source if no valid replacement was found.
                    food_copy = food.copy()
                    food_copy["trial"] = 0
                    new_population.append(food_copy)
            else:
                new_population.append(food)

        return new_population, n_abandoned

    def local_search_phase(self, population: List[dict], progress: float, attempts: int = 5) -> List[dict]:
        """Small memetic improvement around the current global best."""
        if len(population) == 0 or self.global_best_omega is None:
            return population

        best_candidate = None
        scale = max(0.005, 0.04 * (1.0 - progress))

        for _ in range(attempts):
            omega, rho = self._candidate_near_best(scale=scale)
            candidate = self._food_from_arrays(omega, rho)
            if candidate is not None and (best_candidate is None or candidate["score"] > best_candidate["score"]):
                best_candidate = candidate

        if best_candidate is not None:
            worst_idx = int(np.argmin([food["score"] for food in population]))
            if best_candidate["score"] > population[worst_idx]["score"]:
                population[worst_idx] = best_candidate

        return population

    def _inject_elite(self, population: List[dict]) -> List[dict]:
        """Ensure the current global best is not lost."""
        if len(population) == 0 or self.global_best_omega is None:
            return population

        scores = np.array([food["score"] for food in population])
        if np.max(scores) + 1e-15 < self.global_best_score:
            worst_idx = int(np.argmin(scores))
            population[worst_idx] = {
                "omega": self.global_best_omega.copy(),
                "rho": self.global_best_rho.copy(),
                "eff_z": self.global_best_eff_z,
                "eff_y": self.global_best_eff_y,
                "score": self.global_best_score,
                "trial": 0,
            }

        return population

    def optimize(
        self,
        colony_size: int = 50,
        max_iterations: int = 100,
        limit: int = 20,
        verbose: bool = True,
        progress_interval: Optional[int] = None,
        local_search_interval: int = 25,
        local_search_attempts: int = 5,
    ) -> dict:
        """Run the ABC algorithm."""
        start_time = time.time()

        if progress_interval is None:
            # About 20 progress lines per run.
            progress_interval = max(1, max_iterations // 20)

        if verbose:
            print("=" * 80)
            print(f" ABC ALGORITHM - {self.case_name}")
            print("=" * 80)
            print(" Configuration:")
            print(f"   colony_size          = {colony_size}")
            print(f"   max_iterations       = {max_iterations}")
            print(f"   abandonment limit    = {limit}")
            print(f"   objective            = {self.objective}")
            print(f"   progress interval    = every {progress_interval} iterations")
            print(f"   local search interval= every {local_search_interval} iterations")
            print(" Reference P_pi:")
            print(f"   eff_z = {self.eff_z_optimal:.4f}")
            print(f"   eff_y = {self.eff_y_optimal:.4f}\n")

        population = self.initialize_population(colony_size, verbose)

        if len(population) == 0:
            if verbose:
                print("\nFAILED: could not initialize any valid solution.")
            return self._prepare_results(time.time() - start_time, colony_size, 0)

        total_abandoned = 0
        start_best = self.global_best_eff_z

        if verbose:
            header = (
                f"{'iter':>9} | {'best_z':>10} | {'best_y':>10} | "
                f"{'Δstart':>9} | {'vs Pπ':>9} | {'scouts':>6} | {'valid/eval':>11} | {'elapsed':>8}"
            )
            print(header)
            print("-" * len(header))

        for iteration in range(max_iterations):
            progress = (iteration + 1) / max_iterations

            population = self.employed_bee_phase(population, progress)
            population = self.onlooker_bee_phase(population, progress)
            population, n_abandoned = self.scout_bee_phase(population, limit, progress)
            total_abandoned += n_abandoned

            if local_search_interval and (iteration + 1) % local_search_interval == 0:
                population = self.local_search_phase(
                    population,
                    progress,
                    attempts=local_search_attempts,
                )

            population = self._inject_elite(population)

            self.history.append(self.global_best_eff_z)
            self.scout_history.append(n_abandoned)
            self.history_records.append({
                "iteration": iteration + 1,
                "best_eff_z": self.global_best_eff_z,
                "best_eff_y": self.global_best_eff_y,
                "best_score": self.global_best_score,
                "n_abandoned": n_abandoned,
                "eval_count": self.eval_count,
                "valid_count": self.valid_count,
                "elapsed": time.time() - start_time,
            })

            should_print = (
                verbose
                and (
                    iteration == 0
                    or (iteration + 1) % progress_interval == 0
                    or (iteration + 1) == max_iterations
                )
            )

            if should_print:
                delta_start = (
                    100.0 * (self.global_best_eff_z / start_best - 1.0)
                    if start_best > 0 else 0.0
                )
                vs_ppi = (
                    100.0 * (self.global_best_eff_z / self.eff_z_optimal - 1.0)
                    if self.eff_z_optimal > 0 else np.nan
                )
                valid_ratio = f"{self.valid_count}/{self.eval_count}"
                elapsed = time.time() - start_time
                print(
                    f"{iteration+1:9d} | {self.global_best_eff_z:10.4f} | "
                    f"{self.global_best_eff_y:10.4f} | {delta_start:+8.2f}% | "
                    f"{vs_ppi:+8.2f}% | {n_abandoned:6d} | {valid_ratio:>11} | "
                    f"{elapsed:7.1f}s"
                )

        total_time = time.time() - start_time
        return self._prepare_results(total_time, colony_size, total_abandoned)

    def _prepare_results(self, total_time: float, colony_size: int, total_abandoned: int) -> dict:
        relative_to_ppi = (
            100.0 * (self.global_best_eff_z / self.eff_z_optimal - 1.0)
            if self.eff_z_optimal > 0 and self.global_best_eff_z > 0 else -np.inf
        )

        gap_percent = (
            100.0 * (self.eff_z_optimal / self.global_best_eff_z - 1.0)
            if self.eff_z_optimal > 0 and self.global_best_eff_z > 0 else np.inf
        )

        success = self.global_best_eff_z > self.eff_z_optimal

        return {
            "case": self.case_name,
            "best_eff_z": self.global_best_eff_z,
            "best_eff_y": self.global_best_eff_y,
            "best_score": self.global_best_score,
            "best_omega": self.global_best_omega,
            "best_rho": self.global_best_rho,
            "optimal_eff_z": self.eff_z_optimal,
            "optimal_eff_y": self.eff_y_optimal,
            "success": success,
            "gap_percent": gap_percent,
            "relative_to_ppi_percent": relative_to_ppi,
            "improvement_percent": max(relative_to_ppi, 0.0),
            "total_time": total_time,
            "colony_size": colony_size,
            "total_abandoned": total_abandoned,
            "eval_count": self.eval_count,
            "valid_count": self.valid_count,
            "history": self.history.copy(),
            "history_records": self.history_records.copy(),
            "scout_history": self.scout_history.copy(),
            "order": self.order.copy(),
            "enforce_cadsd_order": self.enforce_cadsd_order,
        }

    def print_results(self, results: dict):
        """Print formatted final results."""
        print("=" * 80)
        print(" FINAL RESULTS (ABC)")
        print("=" * 80)
        print(f" Case: {results['case']}")
        print("\n Best solution:")
        print(f"   eff_z = {results['best_eff_z']:.6f}")
        print(f"   eff_y = {results['best_eff_y']:.6f}")
        print("\n Reference P_pi:")
        print(f"   eff_z = {results['optimal_eff_z']:.6f}")
        print(f"   eff_y = {results['optimal_eff_y']:.6f}")
        print("\n Comparison:")
        print(f"   ABC vs P_pi = {results['relative_to_ppi_percent']:+.2f}%")
        print("\n Performance:")
        print(f"   time          = {results['total_time']:.2f}s")
        print(f"   colony size   = {results['colony_size']}")
        print(f"   total scouts  = {results['total_abandoned']}")
        print(f"   valid/eval    = {results['valid_count']}/{results['eval_count']}")

        if len(results["history"]) > 0 and results["history"][0] > 0:
            conv = 100.0 * (results["history"][-1] / results["history"][0] - 1.0)
            print("\n Convergence:")
            print(f"   start eff_z = {results['history'][0]:.4f}")
            print(f"   final eff_z = {results['history'][-1]:.4f}")
            print(f"   gain        = {conv:+.2f}%")
        print("=" * 80)


print("Improved ABCAlgorithm class loaded.")


Improved ABCAlgorithm class loaded.


# Run ABC on MU284


In [6]:
import pandas as pd
import numpy as np

# Read data
df = pd.read_csv('/home/bardia/projects/graphical-sampling/simulations_abc/populations/real/MU284.csv')

# Calculate correlations with P85 (y)
y = df["P85"]

correlations = {
    "P75": np.corrcoef(df["P75"], y)[0, 1],
    "S82": np.corrcoef(df["S82"], y)[0, 1],
    "ME84": np.corrcoef(df["ME84"], y)[0, 1],
    "REV84": np.corrcoef(df["REV84"], y)[0, 1],
    "REG": np.corrcoef(df["REG"], y)[0, 1],
}

print("Correlations with P85 (y):")
print("=" * 40)
for col, corr in sorted(correlations.items(), key=lambda x: abs(x[1]), reverse=True):
    print(f"rho({col}, P85) = {corr:.4f}")

print("\n" + "=" * 40)
print("Mapping to paper notation:")
print("=" * 40)
print("For rho(x,y) ≈ 0.995  -> use x = P75")
print("For rho(z,y) ≈ 0.988  -> use z = ME84")
print("For rho(z,y) ≈ 0.913  -> use z = REV84")
print("For rho(z,y) ≈ 0.865  -> use z = S82")
print("For rho(z,y) ≈ -0.186 -> use z = REG")

print("\n" + "=" * 40)
print("Test combinations structure:")
print("=" * 40)
print("z options: ME84, REV84, REG, S82, Equal")
print("x options: P75, S82, REG, Equal")
print("N_node options: 5, 10, 20, 40")
print(f"Total records: N = {len(df)}")


Correlations with P85 (y):
rho(P75, P85) = 0.9950
rho(ME84, P85) = 0.9875
rho(REV84, P85) = 0.9132
rho(S82, P85) = 0.8654
rho(REG, P85) = -0.1862

Mapping to paper notation:
For rho(x,y) ≈ 0.995  -> use x = P75
For rho(z,y) ≈ 0.988  -> use z = ME84
For rho(z,y) ≈ 0.913  -> use z = REV84
For rho(z,y) ≈ 0.865  -> use z = S82
For rho(z,y) ≈ -0.186 -> use z = REG

Test combinations structure:
z options: ME84, REV84, REG, S82, Equal
x options: P75, S82, REG, Equal
N_node options: 5, 10, 20, 40
Total records: N = 281


In [8]:
import numpy as np
import pandas as pd

print("=" * 80)
print("ABC ON FULL MU284 DATA")
print("=" * 80)

# Load FULL data

# Test cases
test_cases = [
    ("ME84", "P75", "High Corr", 0.988),
    ("S82", "P75", "Medium Corr", 0.865),
    ("REG", "P75", "Low Corr", -0.186),
]

sample_sizes = [5, 10, 20, 40]

y_var = "P85"
N = len(df)

all_results = []
abc_run_details = {}

# A single place to tune all runs.
ABC_RANDOM_SEED = 12345
OBJECTIVE = "eff_z"       # also available: "eff_y", "harmonic", "mean"

for z_name, x_name, corr_label, expected_corr in test_cases:

    print(f"\n{'=' * 80}")
    print(f"{corr_label}: z = {z_name}")
    print(f"{'=' * 80}")

    y_raw = df[y_var].values
    z_raw = df[z_name].values
    x_raw = df[x_name].values

    actual_corr = np.corrcoef(y_raw, z_raw)[0, 1]
    print(f"Corr(P85, {z_name}) = {actual_corr:.3f}")
    print(f"N = {N} (FULL dataset)")

    for n in sample_sizes:
        print(f"\n  n = {n}:")

        # Create pi proportional to x.
        pik_raw = inclusionprobabilities(x_raw, n)

        # Keep the intended ranking from the sampling idea.
        # ABCAlgorithm will internally realign to CaDsd's descending-pi order
        # for mathematically consistent variance evaluation.
        z_over_pi = z_raw / pik_raw
        sort_idx = np.argsort(z_over_pi)

        y_sorted = y_raw[sort_idx]
        z_sorted = z_raw[sort_idx]
        pik_sorted = pik_raw[sort_idx]

        # SRS variances for efficiency ratios.
        var_srs_y = N**2 * (1 - n / N) * np.var(y_raw, ddof=1) / n
        var_srs_z = N**2 * (1 - n / N) * np.var(z_raw, ddof=1) / n

        # ABC parameters. Larger n is harder, but the improved onlooker phase
        # makes the search more effective than the previous version.
        if n == 5:
            colony_size = 180
            max_iterations = 800
            limit = 90
        elif n == 10:
            colony_size = 140
            max_iterations = 650
            limit = 75
        elif n == 20:
            colony_size = 100
            max_iterations = 450
            limit = 55
        else:  # n == 40
            colony_size = 90
            max_iterations = 300
            limit = 45

        progress_interval = max(10, max_iterations // 20)
        local_search_interval = max(10, max_iterations // 20)

        print(
            f"    ABC params: colony={colony_size}, iter={max_iterations}, "
            f"limit={limit}, progress_every={progress_interval}"
        )

        run_key = f"{z_name}_N{N}_n{n}"

        abc = ABCAlgorithm(
            y_sorted=y_sorted,
            z_sorted=z_sorted,
            pik_sorted=pik_sorted,
            var_srs_y=var_srs_y,
            var_srs_z=var_srs_z,
            M=n,
            n=n,
            case_name=run_key,
            objective=OBJECTIVE,
            enforce_cadsd_order=True,
            random_state=ABC_RANDOM_SEED + 1000 * n + len(all_results),
        )

        res = abc.optimize(
            colony_size=colony_size,
            max_iterations=max_iterations,
            limit=limit,
            verbose=True,
            progress_interval=progress_interval,
            local_search_interval=local_search_interval,
            local_search_attempts=5,
        )

        print("\n    Results:")
        print(f"      P_pi(z)        = {res['optimal_eff_z']:9.3f}")
        print(f"      ABC(z)         = {res['best_eff_z']:9.3f}")
        print(f"      ABC vs P_pi    = {res['relative_to_ppi_percent']:+8.2f}%")
        print(f"      valid/eval     = {res['valid_count']}/{res['eval_count']}")
        print(f"      Time           = {res['total_time']:8.1f}s")

        all_results.append({
            "z": z_name,
            "corr": actual_corr,
            "n": n,
            "N": N,
            "ppi_z": res["optimal_eff_z"],
            "abc_z": res["best_eff_z"],
            "ppi_y": res["optimal_eff_y"],
            "abc_y": res["best_eff_y"],
            "abc_vs_ppi_percent": res["relative_to_ppi_percent"],
            "time": res["total_time"],
            "colony": colony_size,
            "iterations": max_iterations,
            "limit": limit,
            "eval_count": res["eval_count"],
            "valid_count": res["valid_count"],
        })

        # Save complete run material for the final sensitivity-analysis section.
        abc_run_details[run_key] = {
            "abc": abc,
            "result": res,
            "sort_idx": sort_idx,
            "z_name": z_name,
            "x_name": x_name,
            "n": n,
            "N": N,
        }

# Final table
print(f"\n\n{'=' * 80}")
print("FINAL SUMMARY TABLE")
print(f"{'=' * 80}\n")

df_res = pd.DataFrame(all_results)

for z in df_res["z"].unique():
    sub = df_res[df_res["z"] == z]
    print(f"\n{z} (rho = {sub['corr'].iloc[0]:.3f})")
    print("-" * 96)
    print(
        f"{'n':>5} {'P_pi(z)':>10} {'ABC(z)':>10} {'ABC-Ppi%':>10} "
        f"{'Time(s)':>9} {'Iter':>6} {'valid/eval':>12}"
    )
    print("-" * 96)
    for _, row in sub.iterrows():
        valid_eval = f"{int(row['valid_count'])}/{int(row['eval_count'])}"
        print(
            f"{row['n']:5.0f} {row['ppi_z']:10.3f} {row['abc_z']:10.3f} "
            f"{row['abc_vs_ppi_percent']:+10.2f} {row['time']:9.1f} "
            f"{row['iterations']:6.0f} {valid_eval:>12}"
        )

# Save compact summary.
df_res.to_csv("abc_mu284_N281_full_improved.csv", index=False)
print("\nSaved: abc_mu284_N281_full_improved.csv")
print("Complete run objects are available in the dictionary: abc_run_details")


ABC ON FULL MU284 DATA

High Corr: z = ME84
Corr(P85, ME84) = 0.988
N = 281 (FULL dataset)

  n = 5:
    ABC params: colony=180, iter=800, limit=90, progress_every=40
 ABC ALGORITHM - ME84_N281_n5
 Configuration:
   colony_size          = 180
   max_iterations       = 800
   abandonment limit    = 90
   objective            = eff_z
   progress interval    = every 40 iterations
   local search interval= every 40 iterations
 Reference P_pi:
   eff_z = 57.7909
   eff_y = 127.2314

 Initializing 180 food sources...


KeyboardInterrupt: 

# Sensitivity Analysis (moved to the end)


In [ ]:
import numpy as np
import pandas as pd


def _abc_variance_from_omega_rho(abc: ABCAlgorithm, omega: np.ndarray, rho: np.ndarray):
    """Return kernel, variances, efficiencies, and validation details for a candidate."""
    try:
        K_dict = CaDsd(pi=abc.pik_sorted, M=abc.M, omega=omega, rho=rho)
        Kmat = K_dict["K"].astype(np.complex128)
    except Exception as e:
        return None, {
            "valid": False,
            "reason": f"CaDsd error: {str(e)[:80]}",
        }

    diag_K = np.real(np.diag(Kmat))
    max_diag_diff = float(np.max(np.abs(diag_K - abc.pik_sorted_desc)))

    if not np.allclose(diag_K, abc.pik_sorted_desc, atol=1e-3):
        return Kmat, {
            "valid": False,
            "reason": f"diagonal mismatch, max diff={max_diag_diff:.3e}",
        }

    try:
        evals = np.linalg.eigvalsh(Kmat)
    except Exception as e:
        return Kmat, {
            "valid": False,
            "reason": f"eigvalsh error: {str(e)[:80]}",
        }

    eig_min = float(evals.min())
    eig_max = float(evals.max())
    trace = float(evals.sum())

    if not (np.all(evals >= -1e-4) and np.all(evals <= 1 + 1e-4)):
        return Kmat, {
            "valid": False,
            "reason": f"eigenvalues outside [0,1], min={eig_min:.3e}, max={eig_max:.3e}",
            "eig_min": eig_min,
            "eig_max": eig_max,
            "trace": trace,
            "max_diag_diff": max_diag_diff,
        }

    if not np.isclose(trace, abc.n, atol=1e-4):
        return Kmat, {
            "valid": False,
            "reason": f"trace mismatch, trace={trace:.6f}, n={abc.n}",
            "eig_min": eig_min,
            "eig_max": eig_max,
            "trace": trace,
            "max_diag_diff": max_diag_diff,
        }

    A = Kmat * (abc.I_N - Kmat.conj())
    var_y = float(np.real(abc.y_sorted.conj().T @ abc.Dpi_inv @ A @ abc.Dpi_inv @ abc.y_sorted))
    var_z = float(np.real(abc.z_sorted.conj().T @ abc.Dpi_inv @ A @ abc.Dpi_inv @ abc.z_sorted))

    eff_y = abc.var_srs_y / var_y if var_y > 0 else np.nan
    eff_z = abc.var_srs_z / var_z if var_z > 0 else np.nan

    return Kmat, {
        "valid": True,
        "reason": "OK",
        "var_y": var_y,
        "var_z": var_z,
        "eff_y": eff_y,
        "eff_z": eff_z,
        "eig_min": eig_min,
        "eig_max": eig_max,
        "trace": trace,
        "max_diag_diff": max_diag_diff,
    }


def sensitivity_analysis_uniform(
    abc: ABCAlgorithm,
    result: dict,
    epsilons=(0.01, 0.02),
    case_name: Optional[str] = None,
    print_table: bool = True,
) -> pd.DataFrame:
    """
    Concise uniform sensitivity analysis around the best ABC design.

    It tests eight perturbation directions for each epsilon:
    omega/rho +/-, omega only +/-, and rho only +/-.
    """
    if result.get("best_omega") is None or result.get("best_rho") is None:
        raise ValueError("The result does not contain best_omega/best_rho. Run ABC first.")

    base_omega = result["best_omega"]
    base_rho = result["best_rho"]
    if case_name is None:
        case_name = result.get("case", abc.case_name)

    _, base_info = _abc_variance_from_omega_rho(abc, base_omega, base_rho)
    if not base_info["valid"]:
        raise ValueError(f"Base design is not valid: {base_info['reason']}")

    perturbation_cases = [
        ("omega+, rho+", +1, +1),
        ("omega+, rho-", +1, -1),
        ("omega-, rho+", -1, +1),
        ("omega-, rho-", -1, -1),
        ("omega+, rho=0", +1, 0),
        ("omega-, rho=0", -1, 0),
        ("omega=0, rho+", 0, +1),
        ("omega=0, rho-", 0, -1),
    ]

    rows = []
    print("\n" + "=" * 90)
    print(f"SENSITIVITY ANALYSIS: {case_name}")
    print("=" * 90)
    print(
        f"Base: var_y={base_info['var_y']:.6g}, var_z={base_info['var_z']:.6g}, "
        f"eff_y={base_info['eff_y']:.4f}, eff_z={base_info['eff_z']:.4f}"
    )

    for eps in epsilons:
        print(f"\nEpsilon = {eps}")
        print("-" * 90)
        print(f"{'case':<18} {'valid':>6} {'d_var_y%':>10} {'d_var_z%':>10} {'d_eff_z%':>10}  reason")
        print("-" * 90)

        for label, s_omega, s_rho in perturbation_cases:
            omega_new = np.clip(base_omega + s_omega * eps, 0.0, 1.0)
            rho_new = np.clip(base_rho + s_rho * eps, 0.0, 1.0)

            _, info = _abc_variance_from_omega_rho(abc, omega_new, rho_new)

            row = {
                "case": case_name,
                "epsilon": eps,
                "perturbation_type": label,
                "valid": info["valid"],
                "reason": info["reason"],
                "base_var_y": base_info["var_y"],
                "base_var_z": base_info["var_z"],
                "base_eff_y": base_info["eff_y"],
                "base_eff_z": base_info["eff_z"],
            }

            if info["valid"]:
                row.update({
                    "var_y": info["var_y"],
                    "var_z": info["var_z"],
                    "eff_y": info["eff_y"],
                    "eff_z": info["eff_z"],
                    "delta_var_y_percent": 100.0 * (info["var_y"] / base_info["var_y"] - 1.0),
                    "delta_var_z_percent": 100.0 * (info["var_z"] / base_info["var_z"] - 1.0),
                    "delta_eff_z_percent": 100.0 * (info["eff_z"] / base_info["eff_z"] - 1.0),
                    "eig_min": info["eig_min"],
                    "eig_max": info["eig_max"],
                    "trace": info["trace"],
                    "max_diag_diff": info["max_diag_diff"],
                })
                print(
                    f"{label:<18} {str(True):>6} "
                    f"{row['delta_var_y_percent']:>+10.3f} "
                    f"{row['delta_var_z_percent']:>+10.3f} "
                    f"{row['delta_eff_z_percent']:>+10.3f}  OK"
                )
            else:
                row.update({
                    "var_y": np.nan,
                    "var_z": np.nan,
                    "eff_y": np.nan,
                    "eff_z": np.nan,
                    "delta_var_y_percent": np.nan,
                    "delta_var_z_percent": np.nan,
                    "delta_eff_z_percent": np.nan,
                })
                print(f"{label:<18} {str(False):>6} {'nan':>10} {'nan':>10} {'nan':>10}  {info['reason']}")

            rows.append(row)

    df_sens = pd.DataFrame(rows)

    if print_table:
        valid_df = df_sens[df_sens["valid"] == True]
        print("\n" + "=" * 90)
        print("SENSITIVITY SUMMARY")
        print("=" * 90)
        print(f"Valid perturbations: {len(valid_df)}/{len(df_sens)}")
        if len(valid_df) > 0:
            print(
                f"Mean delta var_z: {valid_df['delta_var_z_percent'].mean():+.3f}% "
                f"(min {valid_df['delta_var_z_percent'].min():+.3f}%, "
                f"max {valid_df['delta_var_z_percent'].max():+.3f}%)"
            )
            print(
                f"Mean delta eff_z: {valid_df['delta_eff_z_percent'].mean():+.3f}% "
                f"(min {valid_df['delta_eff_z_percent'].min():+.3f}%, "
                f"max {valid_df['delta_eff_z_percent'].max():+.3f}%)"
            )

    return df_sens


print("Sensitivity functions loaded.")


In [ ]:
# Run sensitivity after all ABC runs.
# By default this analyzes only the last completed ABC run, so the output stays concise.
# To analyze another run, set SENSITIVITY_KEY to one key from abc_run_details.keys().

RUN_SENSITIVITY = True
SENSITIVITY_KEY = None      # None -> use the last run
SENSITIVITY_EPSILONS = (0.01, 0.02)

if RUN_SENSITIVITY:
    if "abc_run_details" not in globals() or len(abc_run_details) == 0:
        print("No ABC runs found yet. Run the ABC section first, then run this cell.")
    else:
        if SENSITIVITY_KEY is None:
            SENSITIVITY_KEY = list(abc_run_details.keys())[-1]

        print(f"Available run keys: {list(abc_run_details.keys())}")
        print(f"Running sensitivity for: {SENSITIVITY_KEY}")

        selected = abc_run_details[SENSITIVITY_KEY]
        df_sensitivity = sensitivity_analysis_uniform(
            abc=selected["abc"],
            result=selected["result"],
            epsilons=SENSITIVITY_EPSILONS,
            case_name=SENSITIVITY_KEY,
            print_table=True,
        )

        df_sensitivity.to_csv(f"sensitivity_{SENSITIVITY_KEY}.csv", index=False)
        print(f"\nSaved: sensitivity_{SENSITIVITY_KEY}.csv")
else:
    print("Sensitivity is skipped. Set RUN_SENSITIVITY = True when you want to run it.")
